In [10]:
# Scientific Computing
import pandas as pd
import numpy as np

# Text processing.
import re 
import ftfy
import spacy
import nltk
nltk.download('names')
from nltk import ngrams
from nltk.corpus import names
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# Initializing tok2vec for lemmatization.
# See "Natural Language Processing in Action" for code reference. 
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler"])

[nltk_data] Downloading package names to
[nltk_data]     C:\Users\clara\AppData\Roaming\nltk_data...
[nltk_data]   Package names is already up-to-date!


In [2]:
Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")

C:\Users\clara\AppData\Local\Temp\ipykernel_41028\3248790261.py:1: DtypeWarning: Columns (0: Course_Code, 1: R_Dropin_Tutoring, 2: R_Online_Active_Participation, 3: R_Online_Engaging_Content, 4: R_Weekly_Emails, 5: T_Online_Experience_Suggestions, 6: T_Other_Suggestions) have mixed types. Specify dtype option on import or set low_memory=False.
  Survey_df = pd.read_csv("../Clean_Data_Resources/Survey_Results.csv")


In [3]:
# Define all survey columns that are made of text.
Text_Cols = [
    "T_Collaboration_Understanding",
    "T_Leader_Performance_Suggestions",
    "T_Other_Suggestions",
    "T_Program_Overall_Suggestions",
]

In [4]:
# Tokenize responses.

# Encode responses that functionally do not give information.
Non_Responses = {"n/a", "na", "none", "nothing", "no", "n"}

# Extract nltk's names.
Name_List = set(n.lower() for n in names.words())

def tokenize_response(text):
    # Guard against NaN and empty strings.
    if not isinstance(text, str) or text.strip() == "":
        return []
    # Guard against non-responses.
    if text.strip().lower().replace("/", "") in Non_Responses:
        return []
    # Guard against single character or numeric-only responses.
    if len(text.strip()) <= 1 or text.strip().isdigit():
        return []
    # Fix broken encodings first, then normalize apostrophes and slashes.
    text = ftfy.fix_text(text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("/", " ")
    doc = nlp(text.lower())
    # Strip punctuation, whitespace, contractions, emails, and names.
    return [tok.text for tok in doc
            if not tok.is_punct
            and not tok.is_space
            and tok.text not in {"n't", "ca", "m", "'m", "ve", "'ve", "re", "'re", "'s", "'d"}
            and not re.match(r'\S+@\S+', tok.text)
            and tok.text not in Name_List]

In [5]:
# Use an apply function to tokenize all columns in Text_Cols:
for col in Text_Cols:
    Survey_df[col + "_tokens"] = Survey_df[col].apply(tokenize_response)

c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
c:\Documents\GitHub_Repos\SI_ML_Survey\si_text_env\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger

In [6]:
# Function to enact lemmatization.
def lemmatize_response(tokens):
    if not tokens:
        return []
    doc = nlp(" ".join(tokens))
    return [tok.lemma_ for tok in doc if not tok.is_punct and not tok.is_space]

# Function to enact ngrams, wher n specifies the n...grams. 
def make_ngrams(tokens, n):
    return [" ".join(gram) for gram in ngrams(tokens, n)]

In [7]:
# Lemmatize, then build bigrams and trigrams for each column.
for col in Text_Cols:
    lemma_col = col + "_lemmas"
    Survey_df[lemma_col] = Survey_df[col + "_tokens"].apply(lemmatize_response)

    # Pass n=2 and n=3 to make_ngrams for each lemmatized response.
    Survey_df[col + "_bigrams"]  = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 2))
    Survey_df[col + "_trigrams"] = Survey_df[lemma_col].apply(lambda x: make_ngrams(x, 3))

In [9]:
Survey_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34512 entries, 0 to 34511
Data columns (total 63 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Start_Date                                  34512 non-null  str    
 1   End_Date                                    34512 non-null  str    
 2   Duration                                    34512 non-null  int64  
 3   Recorded_Date                               34512 non-null  str    
 4   Response_ID                                 34512 non-null  str    
 5   Source_File                                 34512 non-null  str    
 6   Discipline                                  34512 non-null  str    
 7   Course_Code                                 34512 non-null  object 
 8   Is_Online                                   34512 non-null  bool   
 9   Semester                                    34512 non-null  str    
 10  Year                 

# VADER Sentiment